# LIGER best - human_pancreas

Seeds: `1-10`

Metrics: BAMI, CASW, SCGRAPHM, ISOLISI, BRAS, KBET, CPCR


In [1]:
import gc
import importlib.util
import os
import sys

os.environ["PYTHONNOUSERSITE"] = "1"
sys.path = [
    path
    for path in sys.path
    if ".local/lib/python3.10/site-packages" not in path
]

PROJECT_ROOT = '/ubc/cs/research/beaver/projects/Single-cell-integration-metrics'
os.environ["R_HOME"] = f"{PROJECT_ROOT}/CondaEnv/pyliger_env/lib/R"
sys.path.insert(0, f"{PROJECT_ROOT}/scib/")

import anndata as ad
import numpy as np
import pandas as pd

OPTIMIZER_PATH = (
    f"{PROJECT_ROOT}/scripts/optuna_multi_metrics/"
    "optuna_liger/optimize_liger_multi_metrics.py"
)
optimizer_spec = importlib.util.spec_from_file_location(
    "liger_opt",
    OPTIMIZER_PATH,
)
if optimizer_spec is None or optimizer_spec.loader is None:
    raise ImportError(f"Cannot load LIGER optimizer: {OPTIMIZER_PATH}")
liger_opt = importlib.util.module_from_spec(optimizer_spec)
sys.modules[optimizer_spec.name] = liger_opt
optimizer_spec.loader.exec_module(liger_opt)


/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/CondaEnv/pyliger_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATASET_NAME = 'human_pancreas'
DATA_PATH = f"{PROJECT_ROOT}/data/{DATASET_NAME}.h5ad"
BATCH_KEY = 'tech'
LABEL_KEY = 'celltype'

SELECTED_BIO_METRICS = ('BAMI', 'CASW', 'SCGRAPHM', 'ISOLISI')
SELECTED_BATCH_METRICS = ('BRAS', 'KBET', 'CPCR')
SELECTED_METRICS = SELECTED_BIO_METRICS + SELECTED_BATCH_METRICS

SEEDS = tuple(range(1, 11))
MODE = "core"
LAYER = "counts"
N_NEIGHBORS = 15
EMBED_KEY = "X_liger"

liger_opt.DATASET_NAME = DATASET_NAME
liger_opt.DATA_PATH = DATA_PATH
liger_opt.BATCH_KEY = BATCH_KEY
liger_opt.LABEL_KEY = LABEL_KEY
liger_opt.MODE = MODE
liger_opt.LAYER = LAYER
liger_opt.USE_BAMI = "BAMI" in SELECTED_METRICS
liger_opt.USE_CASW = "CASW" in SELECTED_METRICS
liger_opt.USE_SCGRAPHM = "SCGRAPHM" in SELECTED_METRICS
liger_opt.USE_TRAJ = "TRAJ" in SELECTED_METRICS
liger_opt.USE_ISOLISI = "ISOLISI" in SELECTED_METRICS
liger_opt.USE_BRAS = "BRAS" in SELECTED_METRICS
liger_opt.USE_KBET = "KBET" in SELECTED_METRICS
liger_opt.USE_CPCR = "CPCR" in SELECTED_METRICS

assert DATASET_NAME == 'human_pancreas'
assert BATCH_KEY == 'tech'
assert LABEL_KEY == 'celltype'
assert SELECTED_METRICS == ('BAMI', 'CASW', 'SCGRAPHM', 'ISOLISI', 'BRAS', 'KBET', 'CPCR')
assert SEEDS == tuple(range(1, 11))
assert liger_opt.CORE_MAX_ITERS == 30
assert liger_opt.N_NEIGHBORS == N_NEIGHBORS
assert liger_opt.EMBED_KEY == EMBED_KEY

pd.Series(
    {
        "dataset": DATASET_NAME,
        "data_path": DATA_PATH,
        "batch_key": BATCH_KEY,
        "label_key": LABEL_KEY,
        "metrics": list(SELECTED_METRICS),
        "seeds": list(SEEDS),
        "mode": MODE,
        "layer": LAYER,
    },
    name="configuration",
)


dataset                                         human_pancreas
data_path    /ubc/cs/research/beaver/projects/Single-cell-i...
batch_key                                                 tech
label_key                                             celltype
metrics      [BAMI, CASW, SCGRAPHM, ISOLISI, BRAS, KBET, CPCR]
seeds                          [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
mode                                                      core
layer                                                   counts
Name: configuration, dtype: object

In [3]:
liger_opt.validate_metric_dependencies()
liger_opt.validate_configuration(list(SELECTED_METRICS))
liger_opt.validate_dataset_contents()


Dataset preflight: n_obs=16382 n_batches=9 n_labels=14
ISOLISI preflight: min_batch_coverage=4 isolated_labels=['t_cell']
CPCR preflight: eligible_labels=9


In [4]:
OPTUNA_STUDY_NAME = 'liger_human_pancreas_joint_expanded_metrics_scgraphm_v1'
OPTUNA_BEST_TRIAL = 31
OPTUNA_BEST_VALUE = 0.7222477364665002
OPTUNA_BEST_PARAMS = {'k': 24, 'value_lambda': 7.1943184973831205}

LIGER_BEST_ARGS = {
    "k": OPTUNA_BEST_PARAMS["k"],
    "value_lambda": OPTUNA_BEST_PARAMS["value_lambda"],
    "max_iters": 30,
    "mode": "core",
    "layer": "counts",
}


def run_liger_best(adata, seed):
    return liger_opt.run_liger(
        adata=adata,
        k=LIGER_BEST_ARGS["k"],
        lam=LIGER_BEST_ARGS["value_lambda"],
        max_iterations=LIGER_BEST_ARGS["max_iters"],
        mode=LIGER_BEST_ARGS["mode"],
        layer=LIGER_BEST_ARGS["layer"],
        seed=seed,
    )


pd.Series(
    {
        "study": OPTUNA_STUDY_NAME,
        "best_trial": OPTUNA_BEST_TRIAL,
        "best_value": OPTUNA_BEST_VALUE,
        **LIGER_BEST_ARGS,
    },
    name="best_setting",
)


study           liger_human_pancreas_joint_expanded_metrics_sc...
best_trial                                                     31
best_value                                               0.722248
k                                                              24
value_lambda                                             7.194318
max_iters                                                      30
mode                                                         core
layer                                                      counts
Name: best_setting, dtype: object

In [5]:
def compute_joint_metrics(adata_raw, adata_int):
    return liger_opt.compute_joint_metrics(adata_raw, adata_int)


In [6]:
results = []

for seed in SEEDS:
    print(f"\n===== seed {seed} =====", flush=True)

    adata_raw = ad.read_h5ad(DATA_PATH)
    adata_int = run_liger_best(adata_raw, seed=seed)
    metric_result = compute_joint_metrics(adata_raw, adata_int)

    row = {"seed": seed, **metric_result}
    results.append(row)
    print(pd.Series(row), flush=True)

    del adata_raw, adata_int
    gc.collect()

results_df = pd.DataFrame(results)
results_df



===== seed 1 =====


/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/CondaEnv/pyliger_env/lib/python3.10/site-packages/numba/np/ufunc/dufunc.py:344: NumbaWarning: Compilation requested for previously compiled argument types ((uint32,)). This has no effect and perhaps indicates a bug in the calling code (compiling a ufunc more than once for the same signature
/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/CondaEnv/pyliger_env/lib/python3.10/site-packages/numba/np/ufunc/dufunc.py:344: NumbaWarning: Compilation requested for previously compiled argument types ((uint32,)). This has no effect and perhaps indicates a bug in the calling code (compiling a ufunc more than once for the same signature
/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/CondaEnv/pyliger_env/lib/python3.10/site-packages/numba/np/ufunc/dufunc.py:344: NumbaWarning: Compilation requested for previously compiled argument types ((uint32,)). This has no effect and perhaps indicates a bug in the

Removing 2011 genes not expressing in celseq.
Removing 2693 genes not expressing in celseq2.


Removing 2724 genes not expressing in fluidigmc1.
Removing 3518 genes not expressing in inDrop1.
Removing 3238 genes not expressing in inDrop2.


Removing 2910 genes not expressing in inDrop3.


...

Cluster for leiden_opt_2.0 with leiden


resolution: 2.0, bAMI: 0.6975400639137476


optimised clustering against liger_metric_label
optimal cluster resolution: 0.8
optimal score: 0.7251083383369786
BRAS...


seed           10.000000
BAMI            0.725108
CASW            0.651695
SCGRAPHM        0.498447
ISOLISI         0.946913
BRAS            0.700734
KBET            0.664942
CPCR            0.887001
bio_score       0.705541
batch_score     0.750892
joint_score     0.723682
dtype: float64


,seed,BAMI,CASW,SCGRAPHM,ISOLISI,BRAS,KBET,CPCR,bio_score,batch_score,joint_score
0,1,0.749521,0.641835,0.482737,0.948630,0.723537,0.666964,0.850793,0.705681,0.747098,0.722248
1,2,0.724227,0.656162,0.394569,0.909228,0.731187,0.638978,0.859166,0.671046,0.743110,0.699872
2,3,0.696086,0.624469,0.424694,0.882693,0.691277,0.608915,0.859246,0.656986,0.719813,0.682116
3,4,0.699250,0.634973,0.461559,0.954033,0.748331,0.640884,0.894333,0.687454,0.761183,0.716945
4,5,0.712611,0.651664,0.391117,0.864614,0.733781,0.682429,0.893991,0.655001,0.770067,0.701028
5,6,0.715620,0.649403,0.401541,0.897554,0.727209,0.643906,0.861324,0.666030,0.744146,0.697276
6,7,0.699431,0.624305,0.396261,0.895558,0.732436,0.662243,0.832170,0.653889,0.742283,0.689246
7,8,0.739332,0.649495,0.381866,0.879271,0.720716,0.644559,0.834376,0.662491,0.733217,0.690781
8,9,0.687182,0.620035,0.538368,0.789276,0.734175,0.617278,0.854880,0.658715,0.735444,0.689407
9,10,0.725108,0.651695,0.498447,0.946913,0.700734,0.664942,0.887001,0.705541,0.750892,0.723682


In [7]:
score_columns = [
    *SELECTED_METRICS,
    "bio_score",
    "batch_score",
    "joint_score",
]

summary_df = pd.DataFrame(
    {
        "mean": results_df[score_columns].mean(),
        "std": results_df[score_columns].std(ddof=0),
        "min": results_df[score_columns].min(),
        "max": results_df[score_columns].max(),
    }
)
summary_df


,mean,std,min,max
BAMI,0.714837,0.018971,0.687182,0.749521
CASW,0.640404,0.012751,0.620035,0.656162
SCGRAPHM,0.437116,0.051730,0.381866,0.538368
ISOLISI,0.896777,0.046619,0.789276,0.954033
BRAS,0.724338,0.015957,0.691277,0.748331
KBET,0.647110,0.021585,0.608915,0.682429
CPCR,0.862728,0.021293,0.832170,0.894333
bio_score,0.672283,0.019087,0.653889,0.705681
batch_score,0.744725,0.013434,0.719813,0.770067
joint_score,0.701260,0.014029,0.682116,0.723682
